In [1]:
from bs4 import BeautifulSoup, Comment
import pandas as pd

import requests
import time
import csv
from io import StringIO

import sys
from pathlib import Path

pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, BASE_URL, HTTP_HEADER
import utils

### [Basketball-Reference Bot Policy](https://www.sports-reference.com/bot-traffic.html)

Basketball reference enforces a 1 day block if you are caught accessing the site more than 20 times a second.  All cells that run loops which collect the data should have `time.sleep(x)` in the loop where x $\geq3$.  In what follows we use `time.sleep(3.3)` for good measure.  Currently the cell should take about 8 minutes to run.

#### COMMENT OUT THIS CELL IF YOU HAVE ALREADY DOWNLOADED THE DATA AND NO CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()` OR YOU WILL BE SAD IF YOU TRY TO RERUN THE NOTEBOOK

In [2]:
utils.scrape_per_game_tables(START_YEAR,END_YEAR)

Scraping data for the 1980-1981 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1981-1982 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1982-1983 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1983-1984 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1984-1985 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1985-1986 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1986-1987 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1987-1988 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1988-1989 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1989-1990 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1990-1991 season...
Adding team win/losses...
Ad

when we import the datafram from the csv it is important to add a comment keyword or it will mislabel the columns using the descriptive comment instead of the names.

Example for the 2025-2026 season

In [3]:
season = '2025-2026'
df = pd.read_csv(str(ROOT)+f'/data/{season}_season/per_game_stats_{season}.csv',encoding='utf-8',comment='#')

In [4]:
df.sort_values('MVP', ascending=False)

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
1,2025,64.0,18.0,Shai Gilgeous-Alexander,27.0,OKC,PG,68.0,68.0,33.2,10.8,19.4,0.553,1.7,4.4,0.386,9.1,15.0,0.602,0.597,7.9,9.0,0.879,0.6,3.7,4.3,6.6,1.4,0.8,2.2,2.0,31.1,0.939
7,2025,54.0,28.0,Nikola Jokić,30.0,DEN,C,65.0,65.0,34.8,9.9,17.4,0.569,1.7,4.5,0.380,8.2,12.9,0.636,0.618,6.1,7.4,0.831,3.0,9.9,12.9,10.7,1.4,0.8,3.7,2.7,27.7,0.634
16,2025,62.0,20.0,Victor Wembanyama,22.0,SAS,C,64.0,55.0,29.2,8.6,16.9,0.512,1.9,5.5,0.349,6.7,11.4,0.590,0.569,5.8,7.0,0.827,2.0,9.5,11.5,3.1,1.0,3.1,2.4,2.4,25.0,0.569
0,2025,53.0,29.0,Luka Dončić,26.0,LAL,PG,64.0,64.0,35.8,10.8,22.8,0.476,4.0,10.8,0.366,6.9,11.9,0.575,0.563,7.9,10.1,0.780,0.6,7.1,7.7,8.3,1.6,0.5,4.0,2.4,33.5,0.250
20,2025,60.0,22.0,Cade Cunningham,24.0,DET,PG,64.0,64.0,33.9,8.6,18.6,0.461,2.0,5.7,0.342,6.6,12.9,0.514,0.513,4.8,6.0,0.812,0.9,4.7,5.5,9.9,1.4,0.8,3.7,3.1,23.9,0.117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,2025,22.0,60.0,Precious Achiuwa,26.0,SAC,C,73.0,57.0,23.9,4.3,8.2,0.528,0.4,1.3,0.278,4.0,6.9,0.577,0.551,1.1,1.9,0.554,2.4,4.3,6.7,1.4,0.9,0.7,0.9,1.7,10.1,0.000
250,2025,42.0,40.0,Derrick Jones Jr.,28.0,LAC,SF,50.0,45.0,27.0,3.8,7.6,0.499,1.1,3.1,0.359,2.7,4.5,0.593,0.571,1.4,1.9,0.763,1.2,2.2,3.5,1.4,0.9,1.0,0.8,2.1,10.1,0.000
251,2025,NaN,NaN,Kevin Huerter,27.0,2TM,SF,69.0,14.0,22.4,3.9,8.6,0.451,1.4,4.7,0.308,2.4,3.9,0.624,0.535,0.8,1.0,0.775,0.5,3.0,3.5,2.6,0.9,0.5,1.0,1.7,10.0,0.000
252,2025,31.0,51.0,Kevin Huerter,27.0,CHI,SF,44.0,11.0,23.6,4.2,9.2,0.455,1.6,5.1,0.314,2.6,4.1,0.630,0.542,0.9,1.3,0.732,0.5,3.3,3.8,2.6,0.8,0.6,1.1,1.9,10.9,0.000


Note that players who have been traded during the season show up multiple times.  One for each team as well as an aggregate row where the team is listed as "#TM" and # is the number of teams they have been on. We have made the team wins/losses for these rows NaNs.  Here are the people who were on three teams.

In [5]:
df.loc[df['Team']=='3TM']

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
428,2025,NaN,NaN,Kyle Anderson,32.0,3TM,SF,43.0,5.0,19.8,2.5,4.7,0.530,0.1,0.3,0.364,2.4,4.4,0.539,0.540,1.1,1.6,0.706,1.0,2.5,3.5,2.9,1.0,0.6,0.9,2.2,6.2,0.0
530,2025,NaN,NaN,RayJ Dennis,24.0,3TM,PG,17.0,0.0,12.1,1.5,4.9,0.313,0.8,2.6,0.311,0.7,2.2,0.316,0.398,0.5,0.5,0.889,0.5,1.1,1.5,1.9,0.2,0.2,1.0,1.2,4.4,0.0
577,2025,NaN,NaN,Patrick Baldwin Jr.,23.0,3TM,SF,9.0,1.0,11.3,1.2,3.0,0.407,0.7,1.6,0.429,0.6,1.4,0.385,0.519,0.6,0.8,0.714,1.3,1.2,2.6,0.6,0.3,0.6,0.6,1.7,3.7,0.0
618,2025,NaN,NaN,Tyus Jones,29.0,3TM,PG,67.0,12.0,14.6,1.1,3.3,0.349,0.6,2.0,0.292,0.5,1.2,0.444,0.440,0.1,0.1,0.778,0.2,1.0,1.1,2.4,0.6,0.0,0.4,0.5,3.0,0.0
641,2025,NaN,NaN,Christian Koloko,25.0,3TM,C,27.0,4.0,13.2,1.1,2.7,0.419,0.0,0.3,0.111,1.1,2.4,0.462,0.426,0.4,0.6,0.667,1.1,1.9,3.0,0.7,0.5,0.9,0.6,1.8,2.7,0.0


Now we import all of the data into a single pandas dataframe

In [6]:
df_tot = utils.player_stat_dataframe(START_YEAR,END_YEAR)

In [7]:
df_tot.sample(10)

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
2985,1987,NaN,NaN,Darryl Dawkins,31.0,2TM,C,6.0,0.0,5.5,0.3,1.5,0.222,0.0,0.0,NaN,0.3,1.5,0.222,0.222,1.0,2.5,0.400,0.3,0.5,0.8,0.3,0.0,0.3,1.2,2.3,1.7,0.0
14827,2010,44.0,38.0,Josh Powell,28.0,ATL,PF,54.0,0.0,12.1,1.7,3.9,0.452,0.0,0.0,0.000,1.7,3.8,0.454,0.452,0.7,0.8,0.800,0.9,1.6,2.5,0.4,0.1,0.1,1.0,1.4,4.1,0.0
8558,1999,29.0,53.0,Chris Whitney,28.0,WAS,PG,82.0,15.0,19.8,2.6,6.4,0.417,1.2,3.1,0.376,1.5,3.2,0.455,0.509,1.4,1.6,0.848,0.2,1.4,1.6,3.8,0.7,0.1,1.3,2.0,7.8,0.0
3351,1988,NaN,NaN,Tony Brown,28.0,2TM,SF,43.0,0.0,8.5,1.2,2.7,0.424,0.1,0.4,0.250,1.1,2.4,0.451,0.441,0.6,0.7,0.774,0.5,0.5,1.0,0.6,0.3,0.1,0.4,1.0,3.0,0.0
10270,2002,38.0,44.0,Óscar Torres,26.0,GSW,SG,17.0,0.0,6.4,0.9,2.1,0.444,0.4,0.8,0.538,0.5,1.4,0.391,0.542,0.8,1.2,0.700,0.2,0.5,0.7,0.2,0.2,0.1,0.5,0.5,3.1,0.0
6416,1995,NaN,NaN,Charles Smith,30.0,2TM,PF,73.0,34.0,23.5,3.3,7.9,0.422,0.0,0.2,0.133,3.3,7.7,0.430,0.424,1.6,2.2,0.730,1.8,3.1,5.0,0.9,0.7,1.1,1.5,3.1,8.3,0.0
16133,2012,54.0,28.0,Solomon Jones,28.0,NYK,PF,2.0,1.0,13.0,0.0,0.5,0.000,0.0,0.0,NaN,0.0,0.5,0.000,0.000,0.0,0.0,NaN,0.5,1.0,1.5,0.0,0.0,0.5,1.5,1.5,0.0,0.0
24780,2025,44.0,38.0,PJ Hall,23.0,CHO,C,12.0,2.0,15.6,2.1,4.2,0.500,0.2,0.2,1.000,1.9,4.0,0.479,0.520,1.8,2.2,0.808,2.5,3.0,5.5,0.7,0.3,0.7,1.3,1.6,6.1,0.0
20860,2020,22.0,50.0,George Hill,34.0,OKC,PG,14.0,14.0,26.4,4.4,8.6,0.508,1.6,4.1,0.386,2.8,4.5,0.619,0.600,1.5,1.8,0.840,0.6,1.5,2.1,3.1,0.9,0.1,0.9,0.9,11.8,0.0
6371,1995,26.0,56.0,Kevin Garnett,19.0,MIN,SF,80.0,43.0,28.7,4.5,9.2,0.491,0.1,0.4,0.286,4.4,8.8,0.499,0.497,1.3,1.9,0.705,2.2,4.1,6.3,1.8,1.1,1.6,1.4,2.4,10.4,0.0


In [8]:
#deduplicating players that show up with MVP shares more than once because they were traded
votes = df_tot[df_tot['MVP'] > 0]
counts = votes.groupby(['Year', 'Player']).agg(
    rows=('Team', 'size'),
    teams=('Team', ', '.join),
    share=('MVP', 'first'),
)
dups = counts[counts['rows'] > 1]
print(f"{len(counts)} vote-getter player-seasons, {len(dups)} counted more than once")
dups

730 vote-getter player-seasons, 8 counted more than once


rows          teams  share
Year Player                                       
1980 George Johnson        2       IND, SAS  0.004
1993 Dominique Wilkins     3  2TM, ATL, LAC  0.001
1994 Clyde Drexler         3  2TM, POR, HOU  0.003
2004 Vince Carter          3  2TM, TOR, NJN  0.002
2008 Chauncey Billups      3  2TM, DET, DEN  0.027
2009 Stephen Jackson       3  2TM, GSW, CHA  0.001
2020 Derrick Rose          3  2TM, DET, NYK  0.010
     James Harden          3  2TM, HOU, BRK  0.001

Except for 1980->George Johnson named 2 separate players on IND and SAS, all others show up twice/thrice because they were traded mid-season

In [9]:
df_tot['is_total'] = df_tot['Team'].str.endswith('TM')
has_tot = df_tot.groupby(['Year', 'Player'])['is_total'].transform('any') # True when a player was traded

# keeping different team rows of traded players so stats are kept, but MVP share moved to the total row only
df_tot['is_partial'] = has_tot & ~df_tot['is_total']
df_tot.loc[df_tot['is_partial'], 'MVP'] = 0.0

#grouping is by name, so two different players with the same name in one season (e.g. George Johnson, 1980) aren't affected here; those shares need a manual fix.
season_view = df_tot[~df_tot['is_partial']]   # one row per player-season: MVP analysis, modeling
team_view   = df_tot[~df_tot['is_total']]     # one row per player-team stint: team-level questions

In [10]:
cols = ['Year', 'Player', 'Team', 'Pos', 'Age', 'G', 'MP', 'PTS', 'TRB', 'BLK', 'MVP']
df_tot.loc[(df_tot['Year'] == 1980) & (df_tot['Player'] == 'George Johnson'), cols]

,Year,Player,Team,Pos,Age,G,MP,PTS,TRB,BLK,MVP
120,1980,George Johnson,IND,PF,24.0,43.0,21.6,10.6,6.5,0.5,0.004
244,1980,George Johnson,SAS,C,32.0,82.0,23.6,5.0,7.3,3.4,0.004


In [11]:
# On the website it shows that the SAS George Johnson got the MVP vote share not the one on IND, so fixing that manually here
df_tot.loc[(df_tot['Year'] == 1980) & (df_tot['Player'] == 'George Johnson') & (df_tot['Team'] == 'IND'), 'MVP'] = 0.0

df_tot.loc[(df_tot['Year'] == 1980) & (df_tot['Player'] == 'George Johnson'), cols]

,Year,Player,Team,Pos,Age,G,MP,PTS,TRB,BLK,MVP
120,1980,George Johnson,IND,PF,24.0,43.0,21.6,10.6,6.5,0.5,0.000
244,1980,George Johnson,SAS,C,32.0,82.0,23.6,5.0,7.3,3.4,0.004


In [12]:
# Rebuild the views now that all MVP fixes (traded players + George Johnson) are applied
season_view = df_tot[~df_tot['is_partial']].copy()
team_view   = df_tot[~df_tot['is_total']].copy()